# ДЗ5. RL для вызова инструментов

Андрей Белкин

Обучение Qwen3.5-2B с GRPO на ToolACE. Strict reward дал BFCL overall **0.7173** против **0.6827** у исходной модели. Partial получил 0.7040, partial с усиленным KL — 0.6787.

[Отчёт PDF](report.pdf)


In [ ]:
from pathlib import Path
import csv
import json
import rl_hw as hw

ROOT = Path(hw.__file__).parent
info = json.loads((ROOT / "outputs/run_info.json").read_text())


## Данные

Только Team-ACE/ToolACE, без SFT. Из 11 300 диалогов подготовлены **10 731 train / 1 165 validation**. Разделение сделано по исходным диалогам; точные дубликаты объединены до split. Отложены 500 нестандартных диалогов, 6 спорных эталонов и 14 длинных примеров. Продолжения после tool в обучение не включены.

Медиана prompt — 701 токен, p99 — 1918. С 350 заданиями BFCL не найдено точных совпадений нормализованных пользовательских реплик; перефразировки и семантическая утечка не проверены. Все 9603 эталона с вызовами прошли round trip через настоящий Qwen template и vLLM parser.


In [ ]:
print("Модель:", info["base_model"])
print("Ревизия:", info["base_revision"])
print("ToolACE revision:", info["toolace"]["revision"])
print("Версии:", info["versions"])
print("Round trip:", info["eda"]["reference_roundtrip"])


## Награды

**Strict:** 1 за совпадение мультимножества вызовов и аргументов, иначе 0. Порядок вызовов и ключей не влияет, повторы сохраняются.

**Partial:** для пары с одинаковым именем `s = 0.25 + 0.75 × F1` по парам «аргумент, значение»; для разных имён — 0. Вызовы сопоставляются один к одному для максимальной суммы `s`. Итог `2 × sum(s) / (Np + Ng)` штрафует лишние и пропущенные вызовы. Вложенное значение сравнивается целиком.

1 и 1.0 равны; bool и строки отличаются от чисел. Optional/default не нормализуются. Пустой, обрезанный или сломанный ответ получает 0. Для no-call нужен непустой ответ без вызовов. Формат проверяется перед штатным Qwen3Coder parser.


In [ ]:
pred = [{"name": "f", "arguments": {"x": 1}},
        {"name": "f", "arguments": {"x": 1}}]
gold = {"kind": "calls", "calls": [{"name": "f", "arguments": {"x": 1.0}}]}
print("Strict:", hw.score_calls(pred, gold, "strict", numeric_equivalence=True))
print("Partial:", hw.score_calls(pred, gold, "partial", numeric_equivalence=True))


## Настройки и обучение

GRPO + LoRA на linear-слоях языковой части: r=16, alpha=32, LR=1e-5, BF16. Batch 1, accumulation 8, 4 rollout на prompt — два исходных prompts на optimizer step. За 32 шага использовано около 64 prompts (0.6% train).

Training/data seed — 42, штатный colocated vLLM rollout seed — 0. Исходная ревизия, данные и остальные настройки одинаковы. На шагах 16/32 чекпойнт выбирается по common strict на одной фиксированной подвыборке из 64 validation-примеров. У всех трёх выбран шаг 32. Итоговая validation оценивает все 1165 примеров. BFCL не участвует в выборе шага.


In [ ]:
for name in ("A_strict", "B_partial", "C_partial_kl"):
    cfg = hw.CONFIGS[name]
    print(name, "reward:", cfg["reward"], "KL:", cfg["beta"],
          "LR:", cfg["learning_rate"], "шагов:", cfg["max_steps"])


### Запуск

```bash
python3 rl_hw.py prepare
python3 rl_hw.py run --output data/runs/grpo
```

Окружение — `Dockerfile`, GPU — A100 80 GB. `run` последовательно выполняет baseline, обучение трёх конфигураций, объединение LoRA и оценку.


## Результаты

| Вариант | KL | BFCL overall | Val strict | Val partial |
|---|---:|---:|---:|---:|
| Baseline | — | 0.6827 | 0.6266 | 0.7998 |
| A strict | 0.02 | 0.7173 | 0.6361 | 0.7903 |
| B partial | 0.02 | 0.7040 | 0.6395 | 0.7980 |
| C partial | 0.10 | 0.6787 | 0.6403 | 0.8063 |

BFCL overall усредняет группы и подкатегории; это не просто correct/350. Evaluator, benchmark и финальные параметры генерации не менялись. Перед оценкой LoRA объединена с исходными весами. Отдельные `metrics.json` и `predictions.jsonl` для четырёх оценок лежат в `outputs/bfcl/`.

![Loss и reward](outputs/loss_reward.png)

Loss относится к GRPO; определения strict и partial reward различаются, поэтому их высоты напрямую не сравниваются.


In [ ]:
with (ROOT / "outputs/results.csv").open() as f:
    for row in csv.DictReader(f):
        print(row["run"], "BFCL:", round(float(row["overall"]), 4),
              "val strict:", round(float(row["validation_strict"]), 4),
              "val partial:", round(float(row["validation_partial"]), 4))


## Ошибки и нулевой advantage

У A основной прирост в parallel: 0.54 → 0.66; irrelevance остался 0.60. A исправил 16 baseline-ошибок и добавил 6 новых; B — 11 и 5; C — 7 и 12.

В `parallel_1` baseline не выдал разобранных вызовов, A вернул два правильных. В `live_multiple_460-145-11` A пропустил `category="Park"`, и правильный ответ baseline стал ошибкой. В `parallel_103` обе модели не выдали ожидаемый инструментальный вызов.

В A 44/64 групп имели одинаковые rollout-награды, в B — 41/64, в C — 38/64. В таких группах relative policy advantage равен нулю; KL всё ещё может менять веса. В 15, 14 и 11 шагах соответственно обе группы имели нулевой advantage. Partial немного уменьшил эту долю, но лучший BFCL в этом запуске остался у strict.

C дал лучшие ToolACE validation strict/partial, но не улучшил BFCL. Средний train KL снизился с 0.000827 у B до 0.000539 у C. Короткий бюджет, один seed и небольшой BFCL не позволяют считать ранжирование устойчивым. Содержание свободного текста no-call отдельно не оценивалось.


In [ ]:
for name, values in info["training_signal"].items():
    print(name, "нулевой advantage:", values["zero_advantage_groups"],
          "/", values["prompt_groups"], "групп; средний KL:", values["mean_kl"])


## Воспроизводимость

Модель и ToolACE закреплены ревизиями. Исходные evaluator и BFCL загружаются по неизменным ссылкам с проверкой SHA256. Данные разделяются по диалогам до выделения ходов. Для выбора шага используется только ToolACE validation.

В `outputs/results.csv` сохранены итоговые метрики, в `training_history.csv` — loss, reward, KL и validation. Версии библиотек, seed, настройки и хеши данных находятся в `run_info.json`. Все 350 предсказаний каждой модели сохранены отдельно в `outputs/bfcl/`. Метрики пересчитаны по этим файлам.

Короткое обучение охватило около 0.6% train. Использован один seed, поэтому устойчивость различий между вариантами не установлена.
